In [0]:
class batchWC():
    def __init__(self):
        self.base_data_dir = "/Volumes/dev/spark_db/volume-2/data"

    def getRawData(self):
        from pyspark.sql.functions import explode, split
        lines = (spark.read
                    .format("text")
                    .option("lineSep", ".")
                    .load(f"{self.base_data_dir}/text")
                )
        return lines.select(explode(split(lines.value, " ")).alias("word"))
    
    def getQualityData(self, rawDF):
        from pyspark.sql.functions import trim, lower
        return ( rawDF.select(lower(trim(rawDF.word)).alias("word"))
                        .where("word is not null")
                        .where("word rlike '[a-z]'")
                )
        
    def getWordCount(self, qualityDF):
        return qualityDF.groupBy("word").count()
    
    def overwriteWordCount(self, wordCountDF):
        ( wordCountDF.write
                    .format("delta")
                    .mode("overwrite")
                    .saveAsTable("dev.spark_db.word_count_table")
        )
    
    def wordCount(self):
        print(f"\tExecuting Word Count...", end='')
        rawDF = self.getRawData()
        qualityDF = self.getQualityData(rawDF)
        resultDF = self.getWordCount(qualityDF)
        self.overwriteWordCount(resultDF)
        print("Done")



In [0]:
class streamWC():
    def __init__(self):
        self.base_data_dir = "/Volumes/dev/spark_db/volume-2/data"

    def getRawData(self):
        from pyspark.sql.functions import explode, split
        lines = (spark.readStream
                    .format("text")
                    .option("lineSep", ".")
                    .load(f"{self.base_data_dir}/text")
                )
        return lines.select(explode(split(lines.value, " ")).alias("word"))
    
    def getQualityData(self, rawDF):
        from pyspark.sql.functions import trim, lower
        return ( rawDF.select(lower(trim(rawDF.word)).alias("word"))
                        .where("word is not null")
                        .where("word rlike '[a-z]'")
                )
        
    def getWordCount(self, qualityDF):
        return qualityDF.groupBy("word").count()
    
    def overwriteWordCount(self, wordCountDF):
        return ( wordCountDF.writeStream
                    .format("delta")
                    .option("checkpointLocation", f"{self.base_data_dir}/chekpoint/word_count")
                    .outputMode("complete")
                    .trigger(availableNow=True)
                    .toTable("dev.spark_db.word_count_table")
                )
    
    def wordCount(self):
        print(f"\tStarting Word Count Stream...", end='')
        rawDF = self.getRawData()
        qualityDF = self.getQualityData(rawDF)
        resultDF = self.getWordCount(qualityDF)
        sQuery = self.overwriteWordCount(resultDF)
        print("Done")
        return sQuery